# Overview

This notebook applies a weighted K-means coreset approach to reduce a large set of trivariate synthetic compound-flooding scenarios into a smaller representative subset.

1. The workflow is applied across different datasets and assumptions, including in-situ/reanalysis data and stationary/time-varying cases. For each location and case, the notebook reads the simulated back-transformed trivariate samples out of Copula framework for each center variable: rainfall (RF), non-tidal residual (NTR), and river discharge (RD).

2. Because K-means is sensitive to variable scale, the trivariate samples are first standardized. K-means clustering is then applied in the normalized RF–RD–NTR space. For each cluster, the original synthetic event closest to the cluster centroid is selected as the representative sample, rather than using the centroid itself. This ensures that the reduced dataset contains physically valid events from the original synthetic sample space.

3. Each representative event is assigned a weight equal to the number of original synthetic samples represented by its cluster. These weights are later used in hydrodynamic simulations to preserve the contribution of the full synthetic event space and find the flood inundation return periods.

4. Finally, the selected representative samples from the RF-centered, NTR-centered, and RD-centered analyses are combined into one reduced dataset. The notebook also generates diagnostic plots comparing the full synthetic sample space with the representative weighted subset in both three-dimensional space and marginal distributions.

In [ ]:

import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances_argmin_min
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (needed for 3D)
import matplotlib.gridspec as gridspec

In [ ]:
locations = ['OceanSprings']
prop = ['Stationary','TimeVarying']
dataset =['','Reanalysis']

sim_ntr_col = 'Sim_NTR'
sim_rf_col = 'Sim_RF'
sim_rd_col = 'Sim_RD'
density_col = 'density'

core_directory = Path(r'C:/Users/sadaf/Desktop/Generating_Boundary_Conditions/POT_24hr_Jan26/Trivariate')


In [ ]:
for data in dataset:
    for loc in locations:
        for p in prop:
            # File paths
            base_path = core_directory / Path(f'{loc}/{data}/{p}')

            if data == '':
                target_events = 5
            else:
                target_events = 10
            df = pd.read_csv(rf'{base_path}\Simulated_Backtransformed_RF_NTR_RD.csv')
            data_select = df[[sim_rf_col, sim_rd_col, sim_ntr_col]]

            # Normalize data first! (K-Means is sensitive to scale)
            data_norm = (data_select - data_select.mean(axis=0)) / data_select.std(axis=0)
            
            inertias = []
            K = range(200, 3000, 200)
            
            for k in K:
                km = KMeans(n_clusters=k, random_state=42, n_init=10)
                km.fit(data_norm)
                inertias.append(km.inertia_)
            
            
            # 1. Run K-Means to find 5000 clusters
            n_subsamples = 1667 # for each variable, so total 5000 samples across 3 variables
            kmeans = KMeans(n_clusters=n_subsamples, random_state=42)
            kmeans.fit(data_norm)
            
            # 2. Find the "Medoids" (The actual sample closest to each cluster center)
            # We don't want to simulate the centroid because it might be a "fake" event.
            closest_indices, _ = pairwise_distances_argmin_min(kmeans.cluster_centers_, data_norm)
            selected_samples = data_select.iloc[closest_indices].copy()
            
            # 3. Calculate Weights (How many original points does this one sample represent?)
            # labels_ contains the cluster ID for each of the 20,000 points
            # unique, counts = np.unique(kmeans.labels_, return_counts=True)
            # weights = counts  # This array aligns with your selected_samples
            weights = np.bincount(kmeans.labels_, minlength=n_subsamples)
            
            # Now, 'selected_samples' is (5000, N) and 'weights' is (5000,)
            # When you run your hydro model, save the result and multiply by 'weights' for stats.
            selected_samples['Weights'] = weights
            selected_samples.to_csv(rf'{base_path}\Space_Representative_Samples_with_Weights_RF.csv')
            ######################################
            df = pd.read_csv(rf'{base_path}\Simulated_Backtransformed_NTR_RF_RD.csv')
            data_select = df[[sim_ntr_col, sim_rd_col, sim_rf_col]]

            # Normalize data first! (K-Means is sensitive to scale)
            data_norm = (data_select - data_select.mean(axis=0)) / data_select.std(axis=0)
            
            # 1. Run K-Means to find 5000 clusters
            n_subsamples = 1667
            kmeans = KMeans(n_clusters=n_subsamples, random_state=42)
            kmeans.fit(data_norm)
            
            # 2. Find the "Medoids" (The actual sample closest to each cluster center)
            # We don't want to simulate the centroid because it might be a "fake" event.
            closest_indices, _ = pairwise_distances_argmin_min(kmeans.cluster_centers_, data_norm)
            selected_samples = data_select.iloc[closest_indices].copy()
            
            # 3. Calculate Weights (How many original points does this one sample represent?)
            # labels_ contains the cluster ID for each of the 20,000 points
            # unique, counts = np.unique(kmeans.labels_, return_counts=True)
            # weights = counts  # This array aligns with your selected_samples
            weights = np.bincount(kmeans.labels_, minlength=n_subsamples)
            
            # Now, 'selected_samples' is (5000, N) and 'weights' is (5000,)
            # When you run your hydro model, save the result and multiply by 'weights' for stats.
            selected_samples['Weights'] = weights
            selected_samples.to_csv(rf'{base_path}\Space_Representative_Samples_with_Weights_NTR.csv')
            ######################################
            df = pd.read_csv(rf'{base_path}\Simulated_Backtransformed_RD_NTR_RF.csv')
            data_select = df[[sim_rf_col, sim_rd_col, sim_ntr_col]]

            # Normalize data first! (K-Means is sensitive to scale)
            data_norm = (data_select - data_select.mean(axis=0)) / data_select.std(axis=0)
            
            # 1. Run K-Means to find 5000 clusters
            n_subsamples = 1667
            kmeans = KMeans(n_clusters=n_subsamples, random_state=42)
            kmeans.fit(data_norm)
            
            # 2. Find the "Medoids" (The actual sample closest to each cluster center)
            # We don't want to simulate the centroid because it might be a "fake" event.
            closest_indices, _ = pairwise_distances_argmin_min(kmeans.cluster_centers_, data_norm)
            selected_samples = data_select.iloc[closest_indices].copy()
            
            # 3. Calculate Weights (How many original points does this one sample represent?)
            # labels_ contains the cluster ID for each of the 20,000 points
            # unique, counts = np.unique(kmeans.labels_, return_counts=True)
            # weights = counts  # This array aligns with your selected_samples
            weights = np.bincount(kmeans.labels_, minlength=n_subsamples)
            
            # Now, 'selected_samples' is (5000, N) and 'weights' is (5000,)
            # When you run your hydro model, save the result and multiply by 'weights' for stats.
            selected_samples['Weights'] = weights
            selected_samples.to_csv(rf'{base_path}\Space_Representative_Samples_with_Weights_RD.csv')


# Showing representative space vs. the entire space
for data in dataset:
    for loc in locations:
        for p in prop:
            # File paths
            base_path = core_directory / Path(f'{loc}/{data}/{p}')
            if data == '':
                target_events = 5
            else:
                target_events = 10
            df_large = pd.read_csv(rf'{base_path}\Simulated_Backtransformed_ALL_Combined_{target_events}_with_Density.csv')
            # df_small = pd.read_csv(rf'{base_path}\Space_Representative_Samples_with_Weights.csv')
            
            file_names = [
                "Space_Representative_Samples_with_Weights_RF.csv",
                "Space_Representative_Samples_with_Weights_NTR.csv",
                "Space_Representative_Samples_with_Weights_RD.csv"
            ]
            
            # Read and concatenate
            dfs = [pd.read_csv(f"{base_path}\{fname}") for fname in file_names]
            df_all = pd.concat(dfs, ignore_index=True)
            df_all.to_csv(rf'{base_path}\Space_Representative_Samples_with_Weights_All.csv', index=False)
            
            df_small = df_all

            xL, yL, zL = df_large[sim_rf_col], df_large[sim_rd_col], df_large[sim_ntr_col]
            xS, yS, zS = df_small[sim_rf_col], df_small[sim_rd_col], df_small[sim_ntr_col]

            # ---- layout ----
            fig = plt.figure(figsize=(9, 7))
            gs = gridspec.GridSpec(
                2, 2,
                width_ratios=[3, 1.4],  # ⬅ increase left column, shrink right
                height_ratios=[2, 3],
                wspace=0.35,
                hspace=0.35
            )
            
            ax_histx = fig.add_subplot(gs[0, 0])                 # top histogram (RF)
            ax_3d    = fig.add_subplot(gs[1, 0], projection='3d')# 3D scatter
            ax_histy = fig.add_subplot(gs[1, 1])                 # right histogram (RD)
            ax_histz = fig.add_subplot(gs[0, 1])                 # top-right histogram (NTR) - compact
            pos = ax_3d.get_position()
            ax_3d.set_position([
                pos.x0 - 0.03,  # ⬅ shift left (try 0.02–0.05)
                pos.y0,
                pos.width,
                pos.height
            ])
            
            # ---- 3D scatter ----
            ax_3d.scatter(xL, zL, yL, s=20, c='blue', alpha=0.75, label='All simulated')
            ax_3d.scatter(xS, zS, yS, s=10, c='hotpink', alpha=0.35, label='Representative')
            
            ax_3d.set_xlabel('RF (mm)')
            ax_3d.set_zlabel('RD (m3/s)')
            ax_3d.set_ylabel('NTR (m)')
            ax_3d.legend(loc='upper left')
            
            # ---- marginal histograms (overlay both datasets) ----
            bins_x = 60
            bins_y = 60
            bins_z = 60
            
            # RF (top)
            ax_histx.hist(xL, bins=bins_x, color='blue', alpha=0.75, density=True)
            ax_histx.hist(xS, bins=bins_x, color='hotpink', alpha=0.55, density=True)
            ax_histx.set_xlabel('RF (mm)')
            ax_histx.set_ylabel('Density')
            ax_histx.tick_params(axis='x', labelbottom=True)
            
            # RD (right) — horizontal
            ax_histy.hist(yL, bins=bins_y, color='blue', alpha=0.75, density=True, orientation='horizontal')
            ax_histy.hist(yS, bins=bins_y, color='hotpink', alpha=0.55, density=True, orientation='horizontal')
            ax_histy.set_ylabel('RD (m3/s)')
            ax_histy.set_xlabel('Density')
            ax_histy.tick_params(axis='y', labelleft=True)
            
            # NTR (top-right) — horizontal
            ax_histz.hist(zL, bins=bins_z, color='blue', alpha=0.75, density=True)
            ax_histz.hist(zS, bins=bins_z, color='hotpink', alpha=0.55, density=True)
            ax_histz.set_xlabel('NTR (m)')
            ax_histz.set_ylabel('Density')
            ax_histz.tick_params(axis='x', labelbottom=True)
            
            
            # ---- match histogram ranges to the scatter ranges (helps interpretation) ----
            ax_histx.set_xlim(min(xL.min(), xS.min()), max(xL.max(), xS.max()))
            ax_histy.set_ylim(min(yL.min(), yS.min()), max(yL.max(), yS.max()))
            ax_histz.set_xlim(min(zL.min(), zS.min()), max(zL.max(), zS.max()))
            plt.savefig(rf'{base_path}/representative_space_vs_entire_space_All.png', dpi = 300)
            plt.show()